# Enhancer vs Promoter Bias Analysis

This notebook analyzes the distribution of enhancer-like and promoter peaks across evolutionary peak sets using HOMER annotation results. The goal is to test the hypothesis that enhancer evolution exceeds promoter evolution.

**Analysis steps:**
1. Import required libraries and define file paths
2. Parse annotated_peaks.txt for regulatory annotation
3. Classify peaks as enhancer-like or promoter
4. Count and compare regulatory types across peak sets

---

## 1. Import Required Libraries and Define Paths

Import the necessary Python libraries and define the paths to the HOMER annotation files for each peak set.

In [ ]:
# Import libraries
import pandas as pd
import os

# Define annotation file paths for each peak set
base_dir = "../HOMER/results"
peak_sets = {
    "shared": os.path.join(base_dir, "shared_peaks_conservative/annotated_peaks.txt"),
    "human_specific": os.path.join(base_dir, "human_specific_peaks_conservative/annotated_peaks.txt"),
    "mouse_specific": os.path.join(base_dir, "mouse_specific_peaks_conservative/annotated_peaks.txt"),
}

peak_sets

## 2. Parse annotated_peaks.txt for Regulatory Annotation

For each peak set, parse the annotation file to extract the regulatory annotation for each peak.

In [ ]:
# Function to parse annotated_peaks.txt and extract annotation column
def parse_annotation(filepath):
    # Try to infer the correct separator and header
    df = pd.read_csv(filepath, sep='\t', comment='#')
    # Find the annotation column (usually named 'Annotation' or similar)
    ann_col = [c for c in df.columns if 'annot' in c.lower() or 'annotation' in c.lower()]
    if len(ann_col) == 0:
        raise ValueError("No annotation column found.")
    return df[ann_col[0]]

# Parse annotation for each peak set
annotations = {k: parse_annotation(v) for k, v in peak_sets.items()}
annotations['shared'].head()

## 3. Classify Peaks as Enhancer-like or Promoter

Classify each peak as 'enhancer-like' if annotated as intergenic or intronic, and as 'promoter' if annotated as TSS.

In [ ]:
# Classify regulatory type for each peak

def classify_regulatory_type(annotation_series):
    ann = annotation_series.str.lower()
    return ann.apply(lambda x: 'enhancer-like' if ('intergenic' in x or 'intron' in x) else ('promoter' if 'tss' in x else 'other'))

reg_types = {k: classify_regulatory_type(v) for k, v in annotations.items()}
reg_types['shared'].value_counts()

## 4. Count and Compare Regulatory Types Across Peak Sets

For each peak set, count the number of enhancer-like and promoter peaks. Compare the distribution to test if enhancer evolution exceeds promoter evolution.

In [ ]:
# Count regulatory types for each peak set
summary = []
for k, v in reg_types.items():
    counts = v.value_counts()
    summary.append({
        'peak_set': k,
        'enhancer-like': counts.get('enhancer-like', 0),
        'promoter': counts.get('promoter', 0),
        'other': counts.get('other', 0)
    })
reg_summary_df = pd.DataFrame(summary)
reg_summary_df

## 5. Biological Insight: Enhancer Evolution Dominates

If the number of enhancer-like peaks is much greater in species-specific sets compared to promoters, this supports the classic conclusion that enhancer evolution exceeds promoter evolution.